Download [dataset](https://www.kaggle.com/competitions/what-on-the-video)

Place in `data/`

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device("cpu")

epochs = 15
batch_size = 16
learning_rate = 1e-3
backbone_learning_rate = 1e-5
warmup_epochs = 2

device

device(type='cuda')

In [2]:
from data_reading import TrainDataset, TestDataset, process_label_csv
from augmentation import VideoTrainTransform, VideoTestTransform
from torch.utils.data import DataLoader

all_labels = process_label_csv("data/train.csv")
val_df = all_labels.sample(frac=0.2, random_state=42)
train_df = all_labels.drop(val_df.index)
print(f"Train: {len(train_df)}, Val: {len(val_df)}")

train_transform = VideoTrainTransform()
val_transform = VideoTestTransform()
test_transform = VideoTestTransform()

train_dataset = TrainDataset(data_path="data/train", labels_df=train_df, transform=train_transform)
val_dataset = TrainDataset(data_path="data/train", labels_df=val_df, transform=val_transform)
test_dataset = TestDataset(data_path="data/test", transform=test_transform)

pin = device.type == "cuda"
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=pin)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, pin_memory=pin)

num_classes = len(train_dataset[0][1])

Train: 232, Val: 58


In [3]:
import math
from itertools import chain

from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor

model = VideoMAEForVideoClassification.from_pretrained(
    "MCG-NJU/videomae-base-finetuned-kinetics",
    num_labels=num_classes,
    ignore_mismatched_sizes=True,
)

optimizer = optim.AdamW(
    [
        {"params": model.videomae.parameters(), "lr": backbone_learning_rate},
        {
            "params": chain(model.fc_norm.parameters(), model.classifier.parameters()),
            "lr": learning_rate,
        },
    ],
)
criterion = nn.BCEWithLogitsLoss()

def scheduler_factory(opt):
    """Linear warmup for warmup_epochs, then cosine decay to 0."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return optim.lr_scheduler.LambdaLR(opt, lr_lambda)



You passed `num_labels=9` which is incompatible to the `id2label` map of length `400`.


Loading weights:   0%|          | 0/162 [00:00<?, ?it/s]

VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.key.bias   | MISSING    |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.value.bias | MISSING   

In [4]:
from trainer import Trainer

trainer = Trainer(model, criterion, optimizer, device, epoch_amount=epochs, scheduler=scheduler_factory)
# trainer.fit(train_loader, val_loader)

In [5]:
# trainer.save("./model.pt")

In [6]:
trainer.load("./model.pt")

In [7]:
val_logits = trainer.predict(val_loader)
val_probs = torch.sigmoid(val_logits)
val_targets = torch.tensor(val_dataset.labels.iloc[:, 1:].values, dtype=torch.float32)

def sample_f1(preds: torch.Tensor, targets: torch.Tensor) -> float:
    """Sample-averaged F1 for multi-label binary predictions."""
    tp = (preds * targets).sum(dim=1)
    fp = (preds * (1 - targets)).sum(dim=1)
    fn = ((1 - preds) * targets).sum(dim=1)
    prec = tp / (tp + fp + 1e-8)
    rec = tp / (tp + fn + 1e-8)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    f1[(tp == 0) & (fp == 0) & (fn == 0)] = 1.0
    return f1.mean().item()

best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.15, 0.85, 0.025):
    binary = (val_probs >= t).float()
    for i in range(binary.shape[0]):
        if binary[i].sum() == 0:
            binary[i, val_probs[i].argmax()] = 1.0
    f1 = sample_f1(binary, val_targets)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = float(t)

print(f"Best threshold: {best_thresh:.3f}  |  Val F1: {best_f1:.4f}")

Best threshold: 0.150  |  Val F1: 1.0000


In [8]:
predictions = trainer.predict(test_loader)

In [9]:
classes = train_dataset.get_classes()
probs = torch.sigmoid(predictions)

print(f"Using threshold: {best_thresh:.3f} (tuned on validation, F1={best_f1:.4f})")

label_col: list[str] = []
for p in probs:
    p = p.float()
    idx = torch.where(p >= best_thresh)[0]
    if idx.numel() == 0:
        idx = p.argmax().unsqueeze(0)
    else:
        order = torch.argsort(p[idx], descending=True)
        idx = idx[order]
    label_col.append(", ".join(classes[int(i)] for i in idx.tolist()))

submission = pd.DataFrame({"file_name": test_dataset.labels, "label": label_col})

print(submission.head())

submission.reset_index().to_csv("data/submission.csv", index=False)

Using threshold: 0.150 (tuned on validation, F1=1.0000)
                                           file_name          label
0  000464896-guatemala-antigua-church-festi_previ...          water
1  000691821-mexico-puerto-vallarta-ocean_preview...          water
2  000692230-panama-canal-clouds-over-gatun_previ...   cloud, water
3    000745494-florida-anhinga-dead-tree_preview.mp4          water
4              000764644-sunset-and-boat_preview.mp4  sunset, water
